**Mounting the google drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Creating project folders**

In [ ]:
import os
os.makedirs('/content/ev_agent/training', exist_ok=True)
os.makedirs('/content/ev_agent/data/processed', exist_ok=True)
os.makedirs('/content/ev_agent/models', exist_ok=True)
print("Folders has been created!")

Folders has been created!


**Installing all required libraries**

In [3]:
!pip install transformers peft trl bitsandbytes accelerate datasets PyYAML torch huggingface_hub sentencepiece scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 65.7 MB/s eta 0:00:00


**Login to Hugging Face**

In [4]:
from huggingface_hub import login
login()

**Copying files from drive to collab**

In [ ]:
import os

# Create folders
os.makedirs('/content/ev_agent/training', exist_ok=True)
os.makedirs('/content/ev_agent/data/processed', exist_ok=True)
os.makedirs('/content/ev_agent/models', exist_ok=True)

# Copy training scripts
!cp /content/drive/MyDrive/Project/ev_agent/training/finetune_sft.py /content/ev_agent/training/
!cp /content/drive/MyDrive/Project/ev_agent/training/finetune_dpo.py /content/ev_agent/training/
!cp /content/drive/MyDrive/Project/ev_agent/training/train_config.yaml /content/ev_agent/training/

# Copy data files
!cp /content/drive/MyDrive/Project/ev_agent/data/processed/train.json /content/ev_agent/data/processed/
!cp /content/drive/MyDrive/Project/ev_agent/data/processed/val.json /content/ev_agent/data/processed/
!cp /content/drive/MyDrive/Project/ev_agent/data/processed/dpo_train.json /content/ev_agent/data/processed/
!cp /content/drive/MyDrive/Project/ev_agent/data/processed/dpo_val.json /content/ev_agent/data/processed/

print("All files has been copied successfully!")

All files has been copied successfully!


**Files verification**

In [ ]:
print("=== Training files ===")
for f in os.listdir('/content/ev_agent/training/'):
    print(f" {f}")

print("\n=== Data files ===")
for f in os.listdir('/content/ev_agent/data/processed/'):
    size = os.path.getsize(f'/content/ev_agent/data/processed/{f}') / 1024 / 1024
    print(f" {f} ({size:.1f} MB)")

=== Training files ===
 finetune_sft.py
 train_config.yaml
 finetune_dpo.py

=== Data files ===
 val.json (2.8 MB)
 dpo_val.json (3.2 MB)
 dpo_train.json (28.5 MB)
 train.json (25.6 MB)


**Fix epoches**

In [ ]:
with open('/content/ev_agent/training/train_config.yaml', 'r') as f:
    content = f.read()

content = content.replace('num_epochs: 3', 'num_epochs: 1')

with open('/content/ev_agent/training/train_config.yaml', 'w') as f:
    f.write(content)

# Verify it changed
with open('/content/ev_agent/training/train_config.yaml', 'r') as f:
    verify = f.read()

if 'num_epochs: 1' in verify:
    print("✅ Successfully changed to 1 epoch!")
else:
    print("❌ Change failed — try again!")

✅ Successfully changed to 1 epoch!


**Code Parameters fixed**

In [ ]:
code = '''
import json
import yaml
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

CONFIG_PATH = Path("/content/ev_agent/training/train_config.yaml")
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

BASE_DIR   = Path("/content/ev_agent")
MODEL_NAME = config["model"]["name"]
TRAIN_FILE = BASE_DIR / "data/processed/train.json"
VAL_FILE   = BASE_DIR / "data/processed/val.json"
OUTPUT_DIR = BASE_DIR / "models/sft_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading data...")
with open(TRAIN_FILE) as f:
    train_data = json.load(f)
with open(VAL_FILE) as f:
    val_data = json.load(f)

def format_sample(s):
    return (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\\n\\n{s['system']}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\\n\\n{s['user']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\\n\\n{s['assistant']}<|eot_id|>"
    )

train_dataset = Dataset.from_list([{"text": format_sample(s)} for s in train_data])
val_dataset   = Dataset.from_list([{"text": format_sample(s)} for s in val_data])
print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")

print("Loading model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False
model.enable_input_require_grads()
print("Model loaded!")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Training {100*trainable/total:.2f}% of parameters")

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    save_steps=100,
    logging_steps=10,
    eval_steps=100,
    eval_strategy="steps",
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    optim="paged_adamw_32bit",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Starting training...")
trainer.train()
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Training complete! Model saved.")
'''

with open("/content/ev_agent/training/finetune_sft.py", "w") as f:
    f.write(code)

print("File rewritten successfully!")

File rewritten successfully!


In [ ]:
# Run this in Colab to check exact versions
!pip show transformers trl peft torch | grep -E "Name|Version"

Name: transformers
Version: 5.16.1
Name: trl
Version: 1.13.0
Name: peft
Version: 0.20.0
Name: torch
Version: 2.11.0+cu128


**Checking the Gpu is running**

In [ ]:
import torch

print("=== GPU Status ===")
print(f"GPU available    : {torch.cuda.is_available()}")
print(f"GPU name         : {torch.cuda.get_device_name(0)}")
print(f"GPU memory total : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"GPU memory used  : {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU memory free  : {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

=== GPU Status ===
GPU available    : True
GPU name         : NVIDIA A100-SXM4-40GB
GPU memory total : 42.4 GB
GPU memory used  : 0.00 GB
GPU memory free  : 0.00 GB


**SFT Training Starting**

In [ ]:
import os
os.chdir('/content/ev_agent')
!python3 training/finetune_sft.py

Streaming output truncated to the last 5000 lines.
 79% 420/529 [01:51<00:29,  3.72it/s]
 80% 421/529 [01:51<00:29,  3.72it/s]
 80% 422/529 [01:52<00:28,  3.72it/s]
 80% 423/529 [01:52<00:28,  3.67it/s]
 80% 424/529 [01:52<00:28,  3.69it/s]
 80% 425/529 [01:53<00:28,  3.65it/s]
 81% 426/529 [01:53<00:28,  3.63it/s]
 81% 427/529 [01:53<00:27,  3.68it/s]
 81% 428/529 [01:53<00:27,  3.69it/s]
 81% 429/529 [01:54<00:26,  3.72it/s]
 81% 430/529 [01:54<00:26,  3.74it/s]
 81% 431/529 [01:54<00:26,  3.74it/s]
 82% 432/529 [01:54<00:25,  3.74it/s]
 82% 433/529 [01:55<00:25,  3.79it/s]
 82% 434/529 [01:55<00:25,  3.73it/s]
 82% 435/529 [01:55<00:25,  3.73it/s]
 82% 436/529 [01:55<00:24,  3.73it/s]
 83% 437/529 [01:56<00:24,  3.72it/s]
 83% 438/529 [01:56<00:24,  3.78it/s]
 83% 439/529 [01:56<00:23,  3.78it/s]
 83% 440/529 [01:57<00:23,  3.76it/s]
 83% 441/529 [01:57<00:23,  3.75it/s]
 84% 442/529 [01:57<00:22,  3.81it/s]
 84% 443/529 [01:57<00:22,  3.78it/s]
 84% 444/529 [01:58<00:22,  3.77it/s]

**Copying DPO Data and loading SFT model from google drive**

In [5]:
import os, shutil
os.makedirs('/content/ev_agent/models', exist_ok=True)
os.makedirs('/content/ev_agent/data/processed', exist_ok=True)
os.makedirs('/content/ev_agent/models/dpo_model', exist_ok=True)

shutil.copytree(
    '/content/drive/MyDrive/Project/ev_agent/models/sft_model/',
    '/content/ev_agent/models/sft_model/',
    dirs_exist_ok=True
)
os.system('cp /content/drive/MyDrive/Project/ev_agent/data/processed/dpo_train.json /content/ev_agent/data/processed/')
os.system('cp /content/drive/MyDrive/Project/ev_agent/data/processed/dpo_val.json /content/ev_agent/data/processed/')
print("Ready for DPO!")

Ready for DPO!


In [7]:
import os

# Create training folder
os.makedirs('/content/ev_agent/training', exist_ok=True)
print("Training folder created!")

Training folder created!


In [8]:
code = '''
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import DPOTrainer, DPOConfig

BASE_DIR   = Path("/content/ev_agent")
SFT_MODEL  = BASE_DIR / "models/sft_model"
OUTPUT_DIR = BASE_DIR / "models/dpo_model"
DPO_TRAIN  = BASE_DIR / "data/processed/dpo_train.json"
DPO_VAL    = BASE_DIR / "data/processed/dpo_val.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading DPO data...")
with open(DPO_TRAIN) as f:
    train_data = json.load(f)
with open(DPO_VAL) as f:
    val_data = json.load(f)

def format_dpo(item):
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\\n\\n{item['system']}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\\n\\n{item['prompt']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\\n\\n"
    )
    return {
        "prompt":   prompt,
        "chosen":   item["chosen"]  + "<|eot_id|>",
        "rejected": item["rejected"] + "<|eot_id|>",
    }

train_dataset = Dataset.from_list([format_dpo(s) for s in train_data])
val_dataset   = Dataset.from_list([format_dpo(s) for s in val_data])
print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")

print("Loading SFT model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(str(SFT_MODEL))
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    str(SFT_MODEL),
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False
model.enable_input_require_grads()
print("SFT model loaded!")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
print("LoRA applied!")

dpo_config = DPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_steps=50,
    save_steps=50,
    logging_steps=10,
    eval_steps=50,
    eval_strategy="steps",
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    optim="paged_adamw_32bit",
    beta=0.1,
    remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

print("Starting DPO training...")
trainer.train()
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("DPO Training complete! Final model saved.")
'''

with open("/content/ev_agent/training/finetune_dpo.py", "w") as f:
    f.write(code)

print("DPO script ready!")

DPO script ready!


In [16]:
with open('/content/ev_agent/training/finetune_dpo.py', 'r') as f:
    content = f.read()

content = content.replace(
    'tokenizer=tokenizer,',
    'processing_class=tokenizer,'
)

with open('/content/ev_agent/training/finetune_dpo.py', 'w') as f:
    f.write(content)

print("✅ Fixed!")

✅ Fixed!


In [ ]:
import os
os.chdir('/content/ev_agent')
!python3 training/finetune_dpo.py

Loading DPO data...
Train: 38,047 | Val: 4,228
Loading SFT model...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:07<00:00, 39.90it/s]
Loading weights: 100% 448/448 [00:00<00:00, 3057.57it/s]
SFT model loaded!
/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
LoRA applied!
Adding EOS to train dataset: 100% 38047/38047 [00:01<00:00, 21721.87 examples/s]
Tokenizing train dataset: 100% 38047/38047 [00:47<00:00, 801.77 examples/s]
Dropping fully truncated examples from train dataset: 100% 380